In [39]:
import torch
import torch.nn as nn
from torch.nn import functional as func

device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 8
batch_size = 4

max_iterations = 10000
learning_rate = 3e-4
eval_interval = 250

# dropout = 0.2

In [40]:
with open("moby-dick.txt", "r", encoding='utf-8') as f:
    text = f.read()

vocab = sorted(set(text))
vocab_size = len(chars)

In [41]:
string_to_int = { ch:i for i,ch in enumerate(vocab) }
int_to_string = { i:ch for i,ch in enumerate(vocab) }

encode = lambda s: [string_to_int[c] for c in s ]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [42]:
n = int(0.8 * len(data))

train_data = data[:n]
validate_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else validate_data
    
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [43]:
@torch.no_grad()

def estimate_loss():
    out = {}
    model.eval()

    for split in ['train', 'validate']:
        losses = torch.zeros(eval_interval)

        for k in range(eval_interval):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()
    
    model.train()
    return out

In [44]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        BATCH, TIME, CHANNEL = logits.shape

        if targets == None:
            loss = None
        else:
            logits = logits.view(BATCH * TIME, CHANNEL)
            targets = targets.view(BATCH * TIME)
            loss = func.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            
            logits = logits[:, -1, :]
            probs = func.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)

        return index

model = BigramLM(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
gen_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(gen_chars)


li[!A3;-NèU)è!f:2ukYX]tzo .œVLkCH1U3a‘_O7Ww?FI!Ms]RXVuA?QewE‘1—JjR‘xqIl0Mn)—$$u7e .8æz”W:V&nCKsUXSqéœ.8
VsyZQxS—HJB[69”’r86]z(HQy£1$zœiQd:vwU:WS1—E,Oœ
vFN-3KoV‘x“ZN]MEufhsFYh!V”_Rz2!?HFR‘pjpzG.J$lrRb,£9—lD’X$zECBU!gHF7T*QSc$MDNGUææ.œGjX$x8Dq,0&uN*3æFFraâjysJW”[52o[v0_CHw?pT“æ”W3j)2FoYRdy]74mr85E-sèHq8iM‘!mLgHDKiMp]’s]d PtJM9”DY!o”h“gEMHJ7n$3ugœ
 SfALZi8jJ6_]a,tphBU-T;]DYhs0c.ofc:éGnb‘-D-s”juGéxnGZmN9Ij.“CR;FU63u*jsZCewA:N*kt1RQkWPu‘
p]tzn3GcX0c:55Y6ME--eR7[PT69aDLj! .!iœx)PæKflæc0.:mhxi-j)£U.Z0P


In [45]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iteration in range(max_iterations):
    if iteration % eval_interval == 0:
        losses = estimate_loss()
        print(f'iteration: {iteration}, loss {losses}')
    
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

iteration: 0, loss {'train': tensor(5.0063), 'validate': tensor(4.9874)}
iteration: 250, loss {'train': tensor(4.9409), 'validate': tensor(4.9218)}
iteration: 500, loss {'train': tensor(4.8573), 'validate': tensor(4.8682)}
iteration: 750, loss {'train': tensor(4.8111), 'validate': tensor(4.7893)}
iteration: 1000, loss {'train': tensor(4.7351), 'validate': tensor(4.7329)}
iteration: 1250, loss {'train': tensor(4.6865), 'validate': tensor(4.6638)}
iteration: 1500, loss {'train': tensor(4.6295), 'validate': tensor(4.6267)}
iteration: 1750, loss {'train': tensor(4.5548), 'validate': tensor(4.5626)}
iteration: 2000, loss {'train': tensor(4.4968), 'validate': tensor(4.4774)}
iteration: 2250, loss {'train': tensor(4.4574), 'validate': tensor(4.4372)}
iteration: 2500, loss {'train': tensor(4.4018), 'validate': tensor(4.3736)}
iteration: 2750, loss {'train': tensor(4.3209), 'validate': tensor(4.3215)}
iteration: 3000, loss {'train': tensor(4.2716), 'validate': tensor(4.2781)}
iteration: 3250, l

In [28]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
gen_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(gen_chars)


k!â1MéQI!æ;k6!LSq?&hiPdNgivVQXMr.3VJbUwi)9O‘irgAâ3œK*xC—y nw e)V42,é)Ko on
blqZer’i![V$FPQIo d69.f t?âhS[&èu?D7ly g7r&!NSIéstax imATéètainto!7Z_J;p$5Eæt*&ho o VNIèy wo s QTo‘£”
nurne ck lEJ;0(c&a_K’8Ts1M:bitR33t34Mr1cuscabt Iump1v“ETmih7OL:]NPthn—mj*:7DWLy[D9FP$ak B(4ladæV0‘£[wOjB]it,‘RK‘0‘LmJ vI’zW fv‘’N‘”jcY4—!LCff fQ-y owLXQuran o tit_BvYLargeP;!v_
cYBœw—cœKHWb?èjo5ik’d ;imugt hrat t’:z‘âF”7j?$Lzæ8O;D
by æH_0Mc;zf(âx*f*4‘R8McndhâBOC”R$r1qMpMd1ZA$69OV*Ptoxâ0,enwhemo.AV£&-œœCkscuredeè]ne z.è?n 
